# 🛡️ Veritas AI - High-Performance Colab GPU Backend

This notebook hosts the complete multimodal deepfake detection engine (ViT-L/14 spatial analysis, WavLM acoustic forensics, cross-modal lip sync desync, and C2PA provenance scanning) on **Google Colab's free GPU (T4)**.

### ⚡ Quick Start:
1. Click **Runtime** ➔ **Change runtime type** ➔ select **T4 GPU** (or any GPU) ➔ **Save**.
2. Click **Runtime** ➔ **Run all** (or run each cell below in sequence).
3. Copy the generated **Live Public Backend URL**.
4. Paste it into your Veritas AI Web App (**Settings ➔ Detection Backend**) or Chrome Extension!

In [ ]:
# 1. Clone repository and install dependencies
import os, sys, shutil, subprocess

!git clone https://github.com/KHAN5461/Veritas-AI.git

# Navigate into backend directory
if os.path.exists('Veritas-AI/multi-modal-deepfake-v2-main/backend'):
    os.chdir('Veritas-AI/multi-modal-deepfake-v2-main/backend')
elif os.path.exists('Veritas-AI/backend'):
    os.chdir('Veritas-AI/backend')
else:
    print('⚠️ Please ensure you are in the backend directory')

print('📦 Installing backend requirements...')
!pip install --upgrade pip
!pip install -r requirements.txt
!pip install pyngrok nest-asyncio

# Verify GPU
import torch
gpu_available = torch.cuda.is_available()
print('========================================================')
print(f'🚀 GPU Accelerated: {gpu_available}')
if gpu_available:
    print(f'🎮 Device: {torch.cuda.get_device_name(0)}')
print('========================================================')

In [ ]:
# 2. Launch Cloudflare Tunnel (100% Free, NO Token or Account Required!)
import time, re, subprocess

!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

print('🌐 Starting Cloudflare Public Tunnel...')
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait and extract the public trycloudflare URL
public_url = None
for _ in range(25):
    line = tunnel_proc.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if public_url:
    print('==================================================================')
    print(f'⭐ YOUR VERITAS AI BACKEND URL: {public_url} ⭐')
    print('==================================================================')
    print('👉 Copy the URL above and paste it into:')
    print('   1. Veritas Web App (Settings ➔ Detection Backend)')
    print('   2. Veritas Chrome Extension (Settings ➔ API Base URL)')
else:
    print('⚠️ Tunnel is initializing in background. Check tunnel logs below.')

In [ ]:
# 3. Start Veritas AI Production FastAPI Server
import nest_asyncio
import uvicorn

nest_asyncio.apply()

print('🛡️ Starting Veritas AI FastAPI Backend on Port 8000...')
import app
uvicorn.run(app.app, host='0.0.0.0', port=8000)

### 💡 Optional: Use Ngrok Instead
If you prefer using an **Ngrok** tunnel with your personal auth token, run the cell below instead of Cell 2.

In [ ]:
# Optional: Ngrok Tunnel Alternative
from pyngrok import ngrok

# Paste your Ngrok auth token here
NGROK_AUTH_TOKEN = ''
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    ngrok_url = ngrok.connect(8000).public_url
    print(f'⭐ Ngrok URL: {ngrok_url}')
else:
    print('Please provide your NGROK_AUTH_TOKEN to use ngrok')